# ACUARIO-IA · Fase 5: flujo de decisiones animado

Esta versión añade un **grafo de flujo del proceso** que muestra visualmente la secuencia: lectura → comparación → decisión → alerta o estado normal → corrección simulada → resultado. Conserva el acuario ilustrado y el grafo de temperaturas.

**Alcance académico:** datos simulados; rango ilustrativo 24–26 °C; sin sensores ni control de dispositivos físicos. La detección usa reglas de umbral y la corrección es una animación educativa, no una recomendación operativa para todas las especies.

In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyBboxPatch, Polygon
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

TEMP_MIN, TEMP_MAX = 24, 26

def diagnosticar(t):
    if t < TEMP_MIN: return 'ALERTA: temperatura baja', 'calentar'
    if t > TEMP_MAX: return 'ALERTA: temperatura alta', 'enfriar'
    return 'NORMAL: temperatura en rango', 'mantener'

def crear_ruta(t):
    ruta = [int(t)]
    while ruta[-1] > TEMP_MAX: ruta.append(ruta[-1] - 1)
    while ruta[-1] < TEMP_MIN: ruta.append(ruta[-1] + 1)
    return ruta

print('ACUARIO-IA listo | rango ilustrativo:', TEMP_MIN, 'a', TEMP_MAX, '°C')

## Simulador
Elige una temperatura y pulsa **Ejecutar**. Observa el acuario, la ruta de temperatura y el **grafo de flujo**, que resalta una etapa a la vez para explicar cómo decide el sistema.

In [ ]:
control_temp = widgets.IntSlider(value=29,min=18,max=32,step=1,description='Temperatura:',continuous_update=False,style={'description_width':'initial'},layout=widgets.Layout(width='420px'))
boton = widgets.Button(description='▶ Ejecutar simulación',button_style='success',icon='play')
salida = widgets.Output()

def ejecutar(_=None):
  with salida:
    clear_output(wait=True)
    inicial=int(control_temp.value); ruta=crear_ruta(inicial); estado,accion=diagnosticar(inicial)
    print(f'Lectura inicial: {inicial} °C | Rango: {TEMP_MIN}–{TEMP_MAX} °C')
    print('Diagnóstico:',estado)
    print('Decisión:', 'mantener monitoreo' if accion=='mantener' else f'alertar; simular acción de {accion}')
    print('Ruta:', ' → '.join(f'{t}°C' for t in ruta))
    print(f'Pasos simulados: {len(ruta)-1} | Temperatura final: {ruta[-1]} °C')
    print('No hay sensores ni actuadores reales conectados.')

    fig=plt.figure(figsize=(15,8),facecolor='#071d33')
    grid=fig.add_gridspec(2,2,height_ratios=[1.05,1],width_ratios=[1,1.35],hspace=.32,wspace=.18)
    ac=fig.add_subplot(grid[:,0]); gr=fig.add_subplot(grid[0,1]); flujo=fig.add_subplot(grid[1,1])
    for ax in (ac,gr,flujo):
      ax.set_facecolor('#071d33'); ax.tick_params(colors='white')
      for sp in ax.spines.values(): sp.set_color('#35536d')

    # Acuario ilustrado con peces móviles
    ac.set_xlim(0,10); ac.set_ylim(0,10); ac.set_xticks([]); ac.set_yticks([]); ac.set_title('ACUARIO SIMULADO',color='white',fontsize=14)
    ac.fill_between([.5,8.1],[1,1],[8.5,8.5],color='#0b6388',alpha=.65)
    ac.plot([.5,.5,8.1,8.1],[1,8.5,8.5,1],color='#76d8ed',lw=2); ac.plot([.5,8.1],[1,1],color='#76d8ed',lw=2)
    for x in [1.2,2.1,7.2]: ac.plot([x,x-.2,x+.15],[1.2,2,2.7],color='#43c59e',lw=3)
    specs=[(2,5.8,'#ffb347',.42),(5,4,'#55d6d2',.35),(3.2,7,'#ff7b9c',.28)]
    bodies=[]; tails=[]; eyes=[]
    for x,y,c,size in specs:
      bodies.append(ac.scatter([x],[y],s=600*size/.4,color=c,edgecolors='white',zorder=4))
      tails.append(ac.scatter([x-.35],[y],s=150*size/.4,marker='>',color=c,zorder=3))
      eyes.append(ac.scatter([x+.12],[y+.06],s=10,color='#071d33',zorder=5))
    ac.add_patch(plt.Rectangle((8.65,2.1),.45,5.2,fill=False,edgecolor='white',lw=2))
    ac.add_patch(plt.Circle((8.875,1.95),.42,color='#f05d5e',ec='white',lw=1.5))
    merc=ac.add_patch(plt.Rectangle((8.79,2.15),.17,1,color='#f05d5e'))
    ttxt=ac.text(5,9.1,'',ha='center',color='white',fontsize=13,fontweight='bold')
    stxt=ac.text(5,.35,'',ha='center',color='#8be9fd',fontsize=10)

    # Grafo de temperaturas: nodos y ruta animada
    estados=list(range(18,33)); gr.set_xlim(17,33); gr.set_ylim(-1.1,2.2); gr.set_yticks([]); gr.set_xticks(estados[::2])
    gr.set_title('GRAFO DE TEMPERATURAS',color='white',fontsize=13); gr.set_xlabel('Temperatura (°C)',color='white')
    gr.axvspan(TEMP_MIN,TEMP_MAX,color='#43c59e',alpha=.2)
    for t in estados[:-1]: gr.annotate('',xy=(t+.85,0),xytext=(t+.15,0),arrowprops={'arrowstyle':'->','color':'#29445d','lw':1})
    gr.scatter(estados,[0]*len(estados),s=65,color='#49647d',zorder=2)
    gr.scatter([TEMP_MIN,TEMP_MAX],[0,0],s=100,color='#43c59e',edgecolors='white',zorder=3)
    linea,=gr.plot([],[],color='#36e0a1',lw=3,marker='o',markersize=7,zorder=5)
    actual=gr.scatter([],[],s=180,color='#ffd166',edgecolors='white',zorder=6)
    detalle=gr.text(25,1.55,'',ha='center',color='white',fontsize=10)

    # Grafo de flujo con etapas que se iluminan durante la animación
    flujo.set_xlim(0,12); flujo.set_ylim(0,5); flujo.set_xticks([]); flujo.set_yticks([])
    flujo.set_title('FLUJO DE DECISIÓN DEL SISTEMA',color='white',fontsize=13)
    etapas=[('1 · Leer\ntemperatura',(1.3,3.2)),('2 · Comparar\ncon rango',(4.0,3.2)),('3 · ¿Dentro\ndel rango?',(6.7,3.2)),('4A · Estado\nnormal',(9.8,4.1)),('4B · Generar\nalerta',(9.8,2.1)),('5 · Corrección\nsimulada',(6.7,.75)),('6 · Resultado\nfinal',(3.5,.75))]
    cajas=[]
    for label,(x,y) in etapas:
      box=FancyBboxPatch((x-.85,y-.42),1.7,.84,boxstyle='round,pad=0.08',facecolor='#17364f',edgecolor='#62839b',linewidth=1.5)
      flujo.add_patch(box); flujo.text(x,y,label,ha='center',va='center',color='white',fontsize=8); cajas.append(box)
    def flecha(a,b,curva=0):
      x1,y1=etapas[a][1]; x2,y2=etapas[b][1]
      flujo.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='->',color='#47677f',lw=1.5,connectionstyle=f'arc3,rad={curva}'))
    for a,b,c in [(0,1,0),(1,2,0),(2,3,-.12),(2,4,.12),(4,5,.12),(5,6,0),(3,6,-.2)]: flecha(a,b,c)
    flujo.text(8.3,4.65,'SÍ',color='#62e6b0',fontsize=9); flujo.text(8.2,2.8,'NO',color='#ff8c69',fontsize=9)
    fase=flujo.text(6,.15,'Preparado para iniciar',ha='center',color='#8be9fd',fontsize=9)

    # Secuencia didáctica: lectura, comparación, decisión y resultado.
    # Si la temperatura ya está en rango, omite la etapa de corrección.
    if accion=='mantener': secuencia=[0,1,2,3,6]
    else: secuencia=[0,1,2,4,5,6]
    frames=len(secuencia)*10
    def actualizar(frame):
      paso=min(frame//10,len(secuencia)-1); etapa_activa=secuencia[paso]
      for i,box in enumerate(cajas):
        box.set_facecolor('#167d75' if i==etapa_activa else '#17364f')
        box.set_edgecolor('#62e6b0' if i==etapa_activa else '#62839b')
      fase.set_text('Etapa activa: '+etapas[etapa_activa][0].replace('\n',' '))
      idx=min(max(0,frame//max(1,frames//max(1,len(ruta)))),len(ruta)-1)
      # Mantener la ruta sincronizada con la etapa de corrección
      if accion=='mantener' or etapa_activa in (0,1,2,3): idx=0
      elif etapa_activa==4: idx=0
      else: idx=min(len(ruta)-1,max(0,(frame-(secuencia.index(5)*10))//2))
      temp=ruta[idx]
      for i,(body,tail,eye,spec) in enumerate(zip(bodies,tails,eyes,specs)):
        x0,y,c,size=spec; x=.9+((x0+frame*.055*(1 if i%2==0 else -1)-.9)%6.8); yy=y+.12*math.sin(frame*.15+i)
        body.set_offsets([[x,yy]]); tail.set_offsets([[x-.35,yy]]); eye.set_offsets([[x+.12,yy+.06]])
      ttxt.set_text(f'Temperatura: {temp} °C')
      if TEMP_MIN<=temp<=TEMP_MAX: stxt.set_text('✓ EN RANGO · monitoreo'); stxt.set_color('#72f1b8')
      else: stxt.set_text('⚠ ALERTA · corrección simulada'); stxt.set_color('#ff7272')
      merc.set_height(max(.15,min(4.5,4.5*(temp-18)/14)))
      xs=ruta[:idx+1]; linea.set_data(xs,[0]*len(xs)); actual.set_offsets([[temp,0]])
      detalle.set_text(f'Ruta: {" → ".join(str(v) for v in xs)} °C')
      return bodies+tails+eyes+[merc,ttxt,stxt,linea,actual,detalle,fase]+cajas

    ani=FuncAnimation(fig,actualizar,frames=frames,interval=220,repeat=True,blit=False)
    plt.close(fig); display(HTML(ani.to_jshtml()))

boton.on_click(ejecutar)
display(widgets.VBox([control_temp,boton,salida]))
ejecutar()

## Cómo explicar esta fase

- **Lectura:** recibe un valor ficticio de temperatura.
- **Comparación:** contrasta la lectura con el intervalo configurado.
- **Decisión:** el nodo se bifurca en normal o alerta.
- **Corrección simulada:** representa pasos de 1 °C hasta entrar en rango; no acciona equipos.
- **Resultado:** muestra la temperatura final y el recorrido.

**Nota técnica:** la lógica principal es una regla de umbral. El grafo de flujo explica las decisiones; el grafo de temperaturas representa estados y transiciones. Esto no debe describirse como un modelo de aprendizaje automático.